# Layer sweep results

Plots the per-layer probe accuracies produced by `layer_sweep.py` (which appends one entry per run to `experimental_outputs/layer_sweep_results.json`).

Accuracy = fraction of statements whose predicted label matches the csv label; datasets are balanced, so chance = 0.5. Pick `probe_layer` where the curves saturate and `intervene_layer` where they start to rise.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

with open('experimental_outputs/layer_sweep_results.json', 'r') as f:
    runs = json.load(f)

# one line per saved run
for i, r in enumerate(runs):
    print(f"[{i}] {r['model']} {r['probe']} train={'+'.join(r['train_datasets'])} "
          f"val={'+'.join(r['val_datasets']) or '-'} seed={r['seed']}")

In [ ]:
# pick a run (default: the most recent)
run = runs[-1]

def run_table(run):
    """One row per layer, one column per accuracy type."""
    rows = {}
    for layer, res in run['results'].items():
        row = {'iid': res['iid']}
        if 'heldout_subjects' in res:
            row['heldout_subjects'] = res['heldout_subjects']
        for dataset, acc in res.get('transfer', {}).items():
            row[f'transfer: {dataset}'] = acc
        rows[int(layer)] = row
    return pd.DataFrame.from_dict(rows, orient='index').sort_index().rename_axis('layer')

table = run_table(run)
table.style.background_gradient(cmap='RdYlGn', vmin=0.5, vmax=1).format('{:.3f}')

In [ ]:
# accuracy vs layer
def plot_run(run, ax=None):
    table = run_table(run)
    ax = ax or plt.figure(figsize=(8, 5)).gca()
    styles = {'iid': 'k-', 'heldout_subjects': 'k--'}
    for column in table.columns:
        style = [styles[column]] if column in styles else []
        ax.plot(table.index, table[column], *style, marker='o', markersize=3, label=column)
    ax.axhline(0.5, color='gray', linewidth=0.5)
    ax.set_xlabel('layer')
    ax.set_ylabel('probe accuracy')
    ax.set_ylim(0.35, 1.02)
    ax.set_title(f"{run['model']}, {run['probe']}, trained on {'+'.join(run['train_datasets'])}")
    ax.legend()
    return ax

plot_run(run)
plt.show()

In [ ]:
# compare runs side by side (e.g. MMProbe vs LRProbe, or different training sets);
# edit the indices to choose which
compare = [runs[i] for i in [-1]]

fig, axes = plt.subplots(1, len(compare), figsize=(7 * len(compare), 5), squeeze=False)
for run, ax in zip(compare, axes[0]):
    plot_run(run, ax=ax)
plt.tight_layout()
plt.show()